In [1]:
import sys
print(sys.executable)

c:\Users\hp\AnomalyDetection\anomaly_venv\Scripts\python.exe


In [2]:
import pandas as pd
import pyodbc
import os
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
from io import StringIO
from scipy.spatial import cKDTree



In [ ]:
# db_path = r"../data/raw/avall.mdb"

# conn = pyodbc.connect(
#     rf"Driver={{Microsoft Access Driver (*.mdb, *.accdb)}};"
#     rf"DBQ={db_path};"
# )

# cursor = conn.cursor()

# for table in cursor.tables(tableType='TABLE'):
#     print(table.table_name)

aircraft
Country
ct_iaids
ct_seqevt
dt_aircraft
dt_events
dt_Flight_Crew
eADMSPUB_DataDictionary
engines
events
Events_Sequence
Findings
Flight_Crew
flight_time
injury
narratives
NTSB_Admin
Occurrences
seq_of_events
states


Converting tables (events, aircraft, flight_crew, narrative) to CSV using Python

In [4]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Tables to extract (4 tables)
tables_to_extract = ['aircraft', 'events', 'Flight_Crew', 'narratives']

# Output directory
output_dir = 'ntsb_csv'
os.makedirs(output_dir, exist_ok=True)

# Connect to MDB database
try:
    # Connection string for Windows
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={os.path.abspath(db_path)};'
    )
    
    conn = pyodbc.connect(conn_str)
    
    # Extract each table
    for i, table in enumerate(tables_to_extract, 1):
        print(f"[{i}/{len(tables_to_extract)}] Extracting table: {table}")
        
        try:
            # Read table into DataFrame
            query = f"SELECT * FROM {table}"
            df = pd.read_sql(query, conn)
            
            # Save to CSV
            output_file = os.path.join(output_dir, f'{table}.csv')
            df.to_csv(output_file, index=False, encoding='utf-8-sig')
            
            # Print summary
            print(f"  File: {output_file}")
            print(f"  Rows: {len(df):,}")
            print(f"  Columns: {len(df.columns)}")
            print(f"  Size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")
            print(f"  Sample columns: {list(df.columns[:5])}")
            print()
            
        except Exception as e:
            print(f"ERROR: {e}")
            print()
    
    conn.close()
    
except pyodbc.Error as e:
    print(f"\nDATABASE CONNECTION ERROR:")
    print(f"  {e}")
    
except Exception as e:
    print(f"\nUNEXPECTED ERROR: {e}")

[1/4] Extracting table: aircraft
  File: ntsb_csv\aircraft.csv
  Rows: 30,726
  Columns: 93
  Size: 10.85 MB
  Sample columns: ['ev_id', 'Aircraft_Key', 'regis_no', 'ntsb_no', 'acft_missing']

[2/4] Extracting table: events
  File: ntsb_csv\events.csv
  Rows: 30,212
  Columns: 73
  Size: 9.38 MB
  Sample columns: ['ev_id', 'ntsb_no', 'ev_type', 'ev_date', 'ev_dow']

[3/4] Extracting table: Flight_Crew
  File: ntsb_csv\Flight_Crew.csv
  Rows: 31,545
  Columns: 33
  Size: 4.55 MB
  Sample columns: ['ev_id', 'Aircraft_Key', 'crew_no', 'crew_category', 'crew_age']

[4/4] Extracting table: narratives
  File: ntsb_csv\narratives.csv
  Rows: 27,852
  Columns: 8
  Size: 125.46 MB
  Sample columns: ['ev_id', 'Aircraft_Key', 'narr_accp', 'narr_accf', 'narr_cause']



Loading tables in Pandas

In [5]:
# Check what was extracted
csv_dir = 'ntsb_csv'
tables = ['events', 'aircraft', 'narratives', 'Flight_Crew']

for table in tables:
    filepath = os.path.join(csv_dir, f'{table}.csv')
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        
        print(f"\n{table.upper()}")
        print(f"Rows: {len(df):,}")
        print(f"Columns: {len(df.columns)}")
        print(f"File size: {os.path.getsize(filepath) / (1024*1024):.2f} MB")
        print(f"\nKey columns:")
        for col in df.columns[:10]:
            print(f"    - {col}")
        
        # Check for critical columns
        if table == 'events':
            critical = ['ev_id', 'ev_date', 'wx_cond_basic', 'ev_highest_injury', 
                       'dec_latitude', 'dec_longitude']
            missing = [c for c in critical if c not in df.columns]
            if missing:
                print(f"\nMissing critical columns: {missing}")
            else:
                print(f"All critical columns present")
        
        # Show sample data
        print(f"\n  Sample data (first row):")
        print(f"  {dict(list(df.iloc[0].items())[:3])}")
        
    else:
        print(f"\n {table}.csv NOT FOUND!")

C:\Users\hp\AppData\Local\Temp\ipykernel_21264\2286730319.py:9: DtypeWarning: Columns (0,10,31,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)



EVENTS
Rows: 30,212
Columns: 73
File size: 9.38 MB

Key columns:
    - ev_id
    - ntsb_no
    - ev_type
    - ev_date
    - ev_dow
    - ev_time
    - ev_tmzn
    - ev_city
    - ev_state
    - ev_country
All critical columns present

  Sample data (first row):
  {'ev_id': '20080211X00175', 'ntsb_no': 'DFW08RA039', 'ev_type': 'ACC'}


C:\Users\hp\AppData\Local\Temp\ipykernel_21264\2286730319.py:9: DtypeWarning: Columns (0,59) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)



AIRCRAFT
Rows: 30,726
Columns: 93
File size: 10.85 MB

Key columns:
    - ev_id
    - Aircraft_Key
    - regis_no
    - ntsb_no
    - acft_missing
    - far_part
    - flt_plan_filed
    - flight_plan_activated
    - damage
    - acft_fire

  Sample data (first row):
  {'ev_id': '20080211X00175', 'Aircraft_Key': np.int64(1), 'regis_no': 'N530NA'}

NARRATIVES
Rows: 27,852
Columns: 8
File size: 125.46 MB

Key columns:
    - ev_id
    - Aircraft_Key
    - narr_accp
    - narr_accf
    - narr_cause
    - narr_inc
    - lchg_date
    - lchg_userid

  Sample data (first row):
  {'ev_id': '20080211X00175', 'Aircraft_Key': np.int64(1), 'narr_accp': 'import'}

FLIGHT_CREW
Rows: 31,545
Columns: 33
File size: 4.55 MB

Key columns:
    - ev_id
    - Aircraft_Key
    - crew_no
    - crew_category
    - crew_age
    - crew_sex
    - crew_city
    - crew_res_state
    - crew_res_country
    - med_certf

  Sample data (first row):
  {'ev_id': '20080107X00026', 'Aircraft_Key': np.int64(1), 'crew_no': 

Joining NTSB tables (events + aircraft + narratives + Flight_Crew)

In [6]:
csv_dir = '../data/ntsb_csv'

#Loading all tables
events = pd.read_csv(
    os.path.join(csv_dir, 'events.csv'), 
    dtype={'ev_id': str}, 
    low_memory=False
)

aircraft = pd.read_csv(
    os.path.join(csv_dir, 'aircraft.csv'), 
    dtype={'ev_id': str}, 
    low_memory=False
)

crew = pd.read_csv(
    os.path.join(csv_dir, 'Flight_Crew.csv'), 
    dtype={'ev_id': str}, 
    low_memory=False
)

narratives = pd.read_csv(
    os.path.join(csv_dir, 'narratives.csv'), 
    dtype={'ev_id': str}, 
    low_memory=False
)

#Merging events table (the main table)
merged = events.copy()

# Merging with aircraft (LEFT JOIN - keep all events)
merged = merged.merge(
    aircraft, 
    on='ev_id', 
    how='left',
    suffixes=('', '_aircraft')
)

# Merging with crew (LEFT JOIN - keep all events)
merged = merged.merge(
    crew, 
    on=['ev_id', 'Aircraft_Key'], 
    how='left',
    suffixes=('', '_crew')
)

# Merging with narratives (LEFT JOIN - keep all events)
merged = merged.merge(
    narratives, 
    on=['ev_id', 'Aircraft_Key'], 
    how='left',
    suffixes=('', '_narr')
)

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, 'ntsb_merged.csv')
merged.to_csv(output_file, index=False)

print(f"  Rows: {len(merged):,}")
print(f"  Columns: {len(merged.columns)}")
print(f"  File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")



  Rows: 37,780
  Columns: 202
  File size: 194.32 MB


In [7]:
dataset = pd.read_csv('../data/processed/ntsb_merged.csv')
print(dataset.columns)

C:\Users\hp\AppData\Local\Temp\ipykernel_21264\1823462427.py:1: DtypeWarning: Columns (0,10,31,51,131,179,190,192) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('../data/processed/ntsb_merged.csv')


Index(['ev_id', 'ntsb_no', 'ev_type', 'ev_date', 'ev_dow', 'ev_time',
       'ev_tmzn', 'ev_city', 'ev_state', 'ev_country',
       ...
       'mr_faa_med_certf', 'pilot_flying', 'available_restraint',
       'restraint_used', 'narr_accp', 'narr_accf', 'narr_cause', 'narr_inc',
       'lchg_date_narr', 'lchg_userid_narr'],
      dtype='object', length=202)


Dropping unused columns

In [8]:
ntsb = pd.read_csv('../data/processed/ntsb_merged.csv', dtype={'ev_id': str}, low_memory=False)

print(f"Before: {len(ntsb.columns)} columns")

# Drop columns that are 100% empty
empty_cols = [col for col in ntsb.columns if ntsb[col].isnull().sum() == len(ntsb)]
print(f"\nDropping {len(empty_cols)} completely empty columns:")
for col in empty_cols[:10]:  # Show first 10
    print(f"  - {col}")

ntsb = ntsb.drop(columns=empty_cols)

print(f"\nAfter dropping empty: {len(ntsb.columns)} columns")

# Save
ntsb.to_csv('../data/processed/ntsb_cleaned.csv', index=False)

Before: 202 columns

Dropping 26 completely empty columns:
  - wx_brief_comp
  - vis_rvv
  - wx_dens_alt
  - wx_int_precip
  - ntsb_docket
  - ntsb_notf_from
  - ntsb_notf_date
  - ntsb_notf_tm
  - fiche_number
  - faa_dist_office

After dropping empty: 176 columns


In [9]:
# Charger les données nettoyées
ntsb = pd.read_csv('../data/processed/ntsb_cleaned.csv', dtype={'ev_id': str}, low_memory=False)

# Convertir date et coordonnées
ntsb['ev_date'] = pd.to_datetime(ntsb['ev_date'], errors='coerce')
ntsb['ev_year'] = ntsb['ev_date'].dt.year
ntsb['dec_latitude'] = pd.to_numeric(ntsb['dec_latitude'], errors='coerce')
ntsb['dec_longitude'] = pd.to_numeric(ntsb['dec_longitude'], errors='coerce')

# Garder seulement les lignes avec coordonnées + date valides
ntsb_geo = ntsb.dropna(subset=['dec_latitude', 'dec_longitude', 'ev_date']).copy()

print(f'Lignes avec coordonnées + date valides : {len(ntsb_geo):,} / {len(ntsb):,}')
print(f'Période : {ntsb_geo["ev_year"].min():.0f} → {ntsb_geo["ev_year"].max():.0f}')

Lignes avec coordonnées + date valides : 34,504 / 37,780
Période : 2008 → 2026


In [10]:
# Créer répertoire cache
cache_dir = '../data/noaa_cache'
os.makedirs(cache_dir, exist_ok=True)

STATIONS_CACHE = os.path.join(cache_dir, 'noaa_stations.csv')

if os.path.exists(STATIONS_CACHE):
    stations = pd.read_csv(STATIONS_CACHE, dtype=str)
else:
    resp = requests.get('https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv', timeout=60)
    stations = pd.read_csv(StringIO(resp.text), dtype=str)
    stations.to_csv(STATIONS_CACHE, index=False)

# Nettoyer colonnes
stations.columns = stations.columns.str.strip()
stations['LAT'] = pd.to_numeric(stations['LAT'], errors='coerce')
stations['LON'] = pd.to_numeric(stations['LON'], errors='coerce')
stations['BEGIN'] = pd.to_numeric(stations['BEGIN'], errors='coerce')
stations['END'] = pd.to_numeric(stations['END'], errors='coerce')

# Filtrer stations US avec coordonnées valides
stations_us = stations[
    stations['CTRY'].str.strip().eq('US') &
    stations['LAT'].notna() & 
    stations['LON'].notna()
].copy()

# Créer station_id (USAF + WBAN sans tiret)
stations_us['station_id'] = (
    stations_us['USAF'].str.strip().str.zfill(6) +
    stations_us['WBAN'].str.strip().str.zfill(5)
)

print(f'Stations US valides : {len(stations_us):,}')
print(f"Exemple station_id : {stations_us['station_id'].head(3).tolist()}")

Stations US valides : 7,074
Exemple station_id : ['62101099999', '62111099999', '62113099999']


In [11]:
def find_nearest_station(ntsb_df, stations_df, max_distance_km=100):
    """Trouve la station météo la plus proche pour chaque accident."""
    
    R = 6371.0  # Rayon de la Terre en km
    
    # Préparer données stations
    sta = stations_df.dropna(subset=['LAT', 'LON']).copy().reset_index(drop=True)
    sta_rad = np.radians(sta[['LAT', 'LON']].values)
    
    # Construire KD-Tree
    tree = cKDTree(sta_rad)
    
    results = []
    total = len(ntsb_df)
    
    print(f"Matching de {total:,} accidents...")
    
    for idx, (_, row) in enumerate(ntsb_df.iterrows()):
        # Chercher 5 stations les plus proches
        query = np.radians([[row['dec_latitude'], row['dec_longitude']]])
        dists, idxs = tree.query(query, k=5)
        
        matched_station = matched_name = matched_dist = None
        
        # Trouver première station dans rayon ET active pendant l'année
        for dist_rad, idx_sta in zip(dists[0], idxs[0]):
            dist_km = dist_rad * R
            
            if dist_km > max_distance_km:
                break
            
            c = sta.iloc[idx_sta]
            
            # Vérifier si station était active
            begin_y = int(str(c['BEGIN'])[:4]) if pd.notna(c['BEGIN']) else 1900
            end_y = int(str(c['END'])[:4]) if pd.notna(c['END']) else 2099
            
            if begin_y <= row['ev_year'] <= end_y:
                matched_station = c['station_id']
                matched_name = c['STATION NAME']
                matched_dist = round(dist_km, 2)
                break
        
        results.append({
            'ev_id': row['ev_id'],
            'noaa_station_id': matched_station,
            'noaa_station_name': matched_name,
            'noaa_dist_km': matched_dist
        })
    
    return pd.DataFrame(results)


# Exécuter le matching
station_matches = find_nearest_station(ntsb_geo, stations_us)

# Statistiques
matched = station_matches['noaa_station_id'].notna().sum()
print(f"Stations trouvées : {matched:,} / {len(station_matches):,} ({matched/len(station_matches)*100:.1f}%)")
print(f"\nDistance (km) :")
print(station_matches['noaa_dist_km'].describe())

# Sauvegarder matches
matches_file = '../data/processed/station_matches.csv'
station_matches.to_csv(matches_file, index=False)

Matching de 34,504 accidents...
Stations trouvées : 31,013 / 34,504 (89.9%)

Distance (km) :
count    31013.000000
mean        17.231835
std         19.889860
min          0.000000
25%          1.000000
50%         10.500000
75%         27.400000
max         99.900000
Name: noaa_dist_km, dtype: float64


In [14]:

import time
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_noaa_for_station_year(station_id, year, cache_dir):
    """Télécharge et cache les données météo d'une station pour une année."""
    cache_file = os.path.join(cache_dir, f'{station_id}_{year}.csv')
    
    # Vérifier cache d'abord
    if os.path.exists(cache_file):
        return (station_id, year, pd.read_csv(cache_file, dtype=str))
    
    # Télécharger depuis NOAA
    url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/{station_id}.csv"
    
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text), dtype=str)
        df.to_csv(cache_file, index=False)
        return (station_id, year, df)
    except:
        return (station_id, year, None)

# Fusionner pour avoir station + année
ntsb_with_stations = ntsb_geo.merge(station_matches, on='ev_id', how='left')

# Liste unique de (station, année)
download_list = ntsb_with_stations[ntsb_with_stations['noaa_station_id'].notna()][
    ['noaa_station_id', 'ev_year']
].drop_duplicates()

print(f"Combinaisons (station, année) : {len(download_list):,}")

weather_cache = {}
failed = []
start_time = time.time()

# Téléchargement parallèle avec 10 workers
with ThreadPoolExecutor(max_workers=10) as executor:
    # Soumettre tous les téléchargements
    futures = {
        executor.submit(download_noaa_for_station_year, row['noaa_station_id'], int(row['ev_year']), cache_dir): 
        (row['noaa_station_id'], int(row['ev_year']))
        for _, row in download_list.iterrows()
    }
    
    completed = 0
    total = len(futures)
    
    # Traiter les résultats au fur et à mesure
    for future in as_completed(futures):
        station_id, year, weather_df = future.result()
        completed += 1
        
        if weather_df is not None:
            key = f"{station_id}_{year}"
            weather_cache[key] = weather_df
        else:
            failed.append((station_id, year))
        
        # Afficher progression tous les 100 fichiers
        if completed % 100 == 0:
            elapsed = time.time() - start_time
            rate = completed / elapsed  # fichiers/seconde
            remaining = (total - completed) / rate / 60  # minutes restantes
            
            print(f"  Téléchargé : {completed:,} / {total:,} ({completed/total*100:.1f}%) "
                  f"| Temps restant : ~{remaining:.1f} min")

total_time = (time.time() - start_time) / 60

print(f"Réussis : {len(weather_cache):,}")
print(f"Échecs : {len(failed):,}")

# NOUVEAU : Sauvegarder weather_cache
cache_file = '../data/processed/weather_cache.pkl'
with open(cache_file, 'wb') as f:
    pickle.dump(weather_cache, f)
print(f"\nweather_cache sauvegardé : {cache_file}")

# Sauvegarder aussi la liste des échecs
failed_file = '../data/processed/failed_downloads.pkl'
with open(failed_file, 'wb') as f:
    pickle.dump(failed, f)
print(f"failed_downloads sauvegardé : {failed_file}")


Combinaisons (station, année) : 15,844
  Téléchargé : 100 / 15,844 (0.6%) | Temps restant : ~6.3 min
  Téléchargé : 200 / 15,844 (1.3%) | Temps restant : ~4.1 min
  Téléchargé : 300 / 15,844 (1.9%) | Temps restant : ~4.9 min
  Téléchargé : 400 / 15,844 (2.5%) | Temps restant : ~4.2 min
  Téléchargé : 500 / 15,844 (3.2%) | Temps restant : ~4.1 min
  Téléchargé : 600 / 15,844 (3.8%) | Temps restant : ~4.1 min
  Téléchargé : 700 / 15,844 (4.4%) | Temps restant : ~4.0 min
  Téléchargé : 800 / 15,844 (5.0%) | Temps restant : ~3.7 min
  Téléchargé : 900 / 15,844 (5.7%) | Temps restant : ~3.9 min
  Téléchargé : 1,000 / 15,844 (6.3%) | Temps restant : ~3.9 min
  Téléchargé : 1,100 / 15,844 (6.9%) | Temps restant : ~3.9 min
  Téléchargé : 1,200 / 15,844 (7.6%) | Temps restant : ~3.7 min
  Téléchargé : 1,300 / 15,844 (8.2%) | Temps restant : ~3.6 min
  Téléchargé : 1,400 / 15,844 (8.8%) | Temps restant : ~3.4 min
  Téléchargé : 1,500 / 15,844 (9.5%) | Temps restant : ~3.3 min
  Téléchargé : 1,60

In [15]:
import pickle
import os

# Charger weather_cache depuis le fichier sauvegardé
cache_file = '../data/processed/weather_cache.pkl'
failed_file = '../data/processed/failed_downloads.pkl'

if os.path.exists(cache_file):
    
    with open(cache_file, 'rb') as f:
        weather_cache = pickle.load(f)
    
    with open(failed_file, 'rb') as f:
        failed = pickle.load(f)
    
else:
    weather_cache = {}
    failed = []

In [41]:
# Recalculer download_list (nécessaire pour les stats)
download_list = ntsb_with_stations[ntsb_with_stations['noaa_station_id'].notna()][
    ['noaa_station_id', 'ev_year']
].drop_duplicates()

# Statistiques téléchargement
total_combinations = len(download_list)
successful_downloads = len(weather_cache)
failed_downloads = len(failed)

print(f"\nTéléchargement :")
print(f"  • Combinaisons (station, année) totales : {total_combinations:,}")
print(f"  • Téléchargements réussis : {successful_downloads:,} ({successful_downloads/total_combinations*100:.1f}%)")
print(f"  • Échecs : {failed_downloads:,} ({failed_downloads/total_combinations*100:.1f}%)")

# Couverture par accident
accidents_with_station = ntsb_with_stations['noaa_station_id'].notna().sum()
print(f"\nAccidents :")
print(f"  • Total accidents : {len(ntsb_with_stations):,}")
print(f"  • Accidents avec station matchée : {accidents_with_station:,} ({accidents_with_station/len(ntsb_with_stations)*100:.1f}%)")

# Estimation de la couverture finale (après merge)
estimated_coverage = successful_downloads / total_combinations * accidents_with_station / len(ntsb_with_stations)
print(f"  • Couverture météo estimée : ~{estimated_coverage*100:.1f}%")

print("\nExemples d'échecs (premiers 10) :")
for station, year in failed[:10]:
    print(f"  • Station {station}, Année {year}")


Téléchargement :
  • Combinaisons (station, année) totales : 15,844
  • Téléchargements réussis : 15,361 (97.0%)
  • Échecs : 483 (3.0%)

Accidents :
  • Total accidents : 55,890
  • Accidents avec station matchée : 51,259 (91.7%)
  • Couverture météo estimée : ~88.9%

Exemples d'échecs (premiers 10) :
  • Station 99819599999, Année 2008
  • Station 72038499999, Année 2008
  • Station 72688799999, Année 2008
  • Station 72033999999, Année 2008
  • Station 72226199999, Année 2008
  • Station 72039399999, Année 2008
  • Station 99733499999, Année 2008
  • Station 99819199999, Année 2008
  • Station 72258013960, Année 2008
  • Station 72058799999, Année 2008


In [ ]:
#FUSION DONNÉES MÉTÉO AVEC NTSB

weather_features = []

for idx, row in ntsb_with_stations.iterrows():
    if (idx + 1) % 5000 == 0:
        print(f"  Traitement : {idx+1:,} / {len(ntsb_with_stations):,}")
    
    station_id = row['noaa_station_id']
    year = row['ev_year']
    date = row['ev_date']
    
    # Initialiser avec None
    weather_row = {
        'noaa_temp_f': None, 'noaa_dewp_f': None, 'noaa_slp_hpa': None,
        'noaa_visib_miles': None, 'noaa_wind_knots': None, 'noaa_maxwind_knots': None,
        'noaa_gust_knots': None, 'noaa_temp_max_f': None, 'noaa_temp_min_f': None,
        'noaa_prcp_in': None, 'noaa_snow_in': None, 'noaa_frshtt': None,
        'noaa_fog': False, 'noaa_rain': False, 'noaa_snow': False,
        'noaa_hail': False, 'noaa_thunder': False, 'noaa_tornado': False,
        'noaa_temp_c': None, 'noaa_temp_max_c': None, 'noaa_temp_min_c': None
    }
    
    # Passer si pas de station
    if pd.isna(station_id):
        weather_features.append(weather_row)
        continue
    
    # Charger données météo
    key = f"{station_id}_{int(year)}"
    if key not in weather_cache:
        weather_features.append(weather_row)
        continue
    
    weather_df = weather_cache[key]
    
    # Trouver date correspondante
    date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
    weather_on_date = weather_df[weather_df['DATE'] == date_str]
    
    if len(weather_on_date) == 0:
        weather_features.append(weather_row)
        continue
    
    # Extraire caractéristiques météo
    w = weather_on_date.iloc[0]
    
    try:
        weather_row['noaa_temp_f'] = pd.to_numeric(w.get('TEMP'), errors='coerce')
        weather_row['noaa_dewp_f'] = pd.to_numeric(w.get('DEWP'), errors='coerce')
        weather_row['noaa_slp_hpa'] = pd.to_numeric(w.get('SLP'), errors='coerce')
        weather_row['noaa_visib_miles'] = pd.to_numeric(w.get('VISIB'), errors='coerce')
        weather_row['noaa_wind_knots'] = pd.to_numeric(w.get('WDSP'), errors='coerce')
        weather_row['noaa_maxwind_knots'] = pd.to_numeric(w.get('MXSPD'), errors='coerce')
        weather_row['noaa_gust_knots'] = pd.to_numeric(w.get('GUST'), errors='coerce')
        weather_row['noaa_temp_max_f'] = pd.to_numeric(w.get('MAX'), errors='coerce')
        weather_row['noaa_temp_min_f'] = pd.to_numeric(w.get('MIN'), errors='coerce')
        weather_row['noaa_prcp_in'] = pd.to_numeric(w.get('PRCP'), errors='coerce')
        weather_row['noaa_snow_in'] = pd.to_numeric(w.get('SNDP'), errors='coerce')
        
        # FRSHTT : Fog/Rain/Snow/Hail/Thunder/Tornado
        frshtt = str(w.get('FRSHTT', '000000'))
        weather_row['noaa_frshtt'] = frshtt
        weather_row['noaa_fog'] = frshtt[0] == '1'
        weather_row['noaa_rain'] = frshtt[1] == '1'
        weather_row['noaa_snow'] = frshtt[2] == '1'
        weather_row['noaa_hail'] = frshtt[3] == '1'
        weather_row['noaa_thunder'] = frshtt[4] == '1'
        weather_row['noaa_tornado'] = frshtt[5] == '1'
        
        # Conversion °F → °C
        if weather_row['noaa_temp_f'] is not None:
            weather_row['noaa_temp_c'] = (weather_row['noaa_temp_f'] - 32) * 5/9
        if weather_row['noaa_temp_max_f'] is not None:
            weather_row['noaa_temp_max_c'] = (weather_row['noaa_temp_max_f'] - 32) * 5/9
        if weather_row['noaa_temp_min_f'] is not None:
            weather_row['noaa_temp_min_c'] = (weather_row['noaa_temp_min_f'] - 32) * 5/9
    except:
        pass
    
    weather_features.append(weather_row)

# Convertir en DataFrame
weather_df = pd.DataFrame(weather_features)

# Fusionner avec NTSB
ntsb_enriched = pd.concat([ntsb_with_stations.reset_index(drop=True), weather_df], axis=1)

print(f"\nFusion terminée !")
print(f"Dataset final : {len(ntsb_enriched):,} lignes × {len(ntsb_enriched.columns)} colonnes")

  Traitement : 5,000 / 55,890
  Traitement : 10,000 / 55,890
  Traitement : 15,000 / 55,890
  Traitement : 20,000 / 55,890
  Traitement : 25,000 / 55,890
  Traitement : 30,000 / 55,890
  Traitement : 35,000 / 55,890
  Traitement : 40,000 / 55,890
  Traitement : 45,000 / 55,890
  Traitement : 50,000 / 55,890
  Traitement : 55,000 / 55,890

✓ Fusion terminée !
Dataset final : 55,890 lignes × 200 colonnes


In [18]:
# Total d'accidents
total_accidents = len(ntsb_enriched)

# Accidents avec station matchée
accidents_with_station = ntsb_enriched['noaa_station_id'].notna().sum()

# Accidents avec données météo réelles (température non nulle)
accidents_with_weather = ntsb_enriched['noaa_temp_f'].notna().sum()

# Accidents SANS station
accidents_without_station = ntsb_enriched['noaa_station_id'].isna().sum()

# Accidents avec station MAIS sans données météo
accidents_station_but_no_data = (
    ntsb_enriched['noaa_station_id'].notna() & 
    ntsb_enriched['noaa_temp_f'].isna()
).sum()

print(f"  • Total accidents : {total_accidents:,}")
print(f"  • Accidents avec station matchée : {accidents_with_station:,} ({accidents_with_station/total_accidents*100:.1f}%)")
print(f"  • Accidents avec données météo : {accidents_with_weather:,} ({accidents_with_weather/total_accidents*100:.1f}%)")

print(f"  • Accidents sans station (trop loin) : {accidents_without_station:,} ({accidents_without_station/total_accidents*100:.1f}%)")
print(f"  • Accidents avec station mais données manquantes : {accidents_station_but_no_data:,} ({accidents_station_but_no_data/total_accidents*100:.1f}%)")

print(f"\nCOUVERTURE FINALE : {accidents_with_weather/total_accidents*100:.1f}%")

# Détail par colonne météo
print("\nCouverture par feature météo :")
weather_cols = ['noaa_temp_f', 'noaa_wind_knots', 'noaa_visib_miles', 'noaa_fog', 'noaa_rain']
for col in weather_cols:
    coverage = ntsb_enriched[col].notna().sum() if col in ['noaa_temp_f', 'noaa_wind_knots', 'noaa_visib_miles'] else (ntsb_enriched[col] == True).sum()
    pct = coverage / total_accidents * 100
    print(f"  • {col:20} : {coverage:>6,} / {total_accidents:,} ({pct:5.1f}%)")

  • Total accidents : 55,890
  • Accidents avec station matchée : 51,259 (91.7%)
  • Accidents avec données météo : 48,020 (85.9%)
  • Accidents sans station (trop loin) : 4,631 (8.3%)
  • Accidents avec station mais données manquantes : 3,239 (5.8%)

COUVERTURE FINALE : 85.9%

Couverture par feature météo :
  • noaa_temp_f          : 48,020 / 55,890 ( 85.9%)
  • noaa_wind_knots      : 48,020 / 55,890 ( 85.9%)
  • noaa_visib_miles     : 48,020 / 55,890 ( 85.9%)
  • noaa_fog             :  3,372 / 55,890 (  6.0%)
  • noaa_rain            : 11,378 / 55,890 ( 20.4%)


In [21]:
# Filtrer : garder seulement les accidents avec météo complète
ntsb_final = ntsb_enriched[ntsb_enriched['noaa_temp_f'].notna()].copy()

print(f"Dataset complet : {len(ntsb_enriched):,} accidents")
print(f"Dataset filtré (avec météo) : {len(ntsb_final):,} accidents")
print(f"Exclus (sans météo) : {len(ntsb_enriched) - len(ntsb_final):,} accidents ({(len(ntsb_enriched) - len(ntsb_final))/len(ntsb_enriched)*100:.1f}%)")
print(f"Colonnes : {len(ntsb_final.columns)}")

# Sauvegarder le dataset final
output_file = '../data/processed/ntsb_noaa_final.csv'
ntsb_final.to_csv(output_file, index=False)


Dataset complet : 55,890 accidents
Dataset filtré (avec météo) : 48,020 accidents
Exclus (sans météo) : 7,870 accidents (14.1%)
Colonnes : 200


Check for missing values in the dataset

In [25]:
# Rebuild full missing stats (all columns)
missing_all = pd.DataFrame({
    'column': ntsb_final.columns,
    'missing_count': ntsb_final.isnull().sum().values,
    'missing_pct': (ntsb_final.isnull().sum() / len(ntsb_final) * 100).round(2).values,
    'dtype': ntsb_final.dtypes.values
}).sort_values('missing_pct', ascending=False)

# Tier classification
def classify_tier(pct):
    if pct == 0:    return 'COMPLETE'
    elif pct >= 80: return 'DROP (>80%)'
    elif pct >= 50: return 'REVIEW (50-80%)'
    elif pct >= 20: return 'IMPUTE (20-50%)'
    else:           return 'EASY_IMPUTE (<20%)'

missing_all['tier'] = missing_all['missing_pct'].apply(classify_tier)

# Summary by tier
print("TIERED MISSING VALUE ANALYSIS")
for tier in ['COMPLETE', 'EASY_IMPUTE (<20%)', 'IMPUTE (20-50%)', 
             'REVIEW (50-80%)', 'DROP (>80%)']:
    subset = missing_all[missing_all['tier'] == tier]
    print(f"\n{tier}: {len(subset)} columns")
    if tier != 'COMPLETE':
        for _, row in subset.iterrows():
            print(f"  {row['column']:30} {row['missing_pct']:6.1f}%  [{row['dtype']}]")

complete_cols = missing_all[missing_all['tier'] == 'COMPLETE']['column'].tolist()
print(f"THE {len(complete_cols)} COMPLETE COLUMNS (0% missing):")

for i, col in enumerate(complete_cols, 1):
    sample_vals = ntsb_final[col].value_counts().head(3).to_dict()
    print(f"  {i:2}. {col:35} dtype={ntsb_final[col].dtype}")
    print(f"      Sample values: {sample_vals}")

TIERED MISSING VALUE ANALYSIS

COMPLETE: 68 columns

EASY_IMPUTE (<20%): 66 columns
  dest_country                     19.8%  [object]
  latlong_acq                      19.7%  [object]
  afm_hrs                          19.0%  [float64]
  dprt_apt_id                      18.7%  [object]
  date_last_insp                   18.0%  [object]
  sky_cond_nonceil                 17.9%  [object]
  cert_max_gr_wt                   16.0%  [float64]
  sky_cond_ceil                    15.9%  [object]
  wx_obs_elev                      15.4%  [float64]
  type_fly                         12.8%  [object]
  elt_install                      12.1%  [object]
  dprt_state                       12.0%  [object]
  crew_res_state                   11.5%  [object]
  crew_city                        11.2%  [object]
  dprt_city                        11.1%  [object]
  dprt_country                     10.8%  [object]
  pc_profession                    10.8%  [object]
  type_last_insp                   10.0%  [obj

Dropping >80% missing columns

In [26]:
cols_to_drop = missing_all[missing_all['tier'] == 'DROP (>80%)']['column'].tolist()

print(f"Dropping {len(cols_to_drop)} columns (>80% missing):")
for col in cols_to_drop:
    pct = missing_all[missing_all['column'] == col]['missing_pct'].values[0]
    print(f"  ✗ {col:30} ({pct:.1f}% missing)")

ntsb_final = ntsb_final.drop(columns=cols_to_drop)

print(f"\nDataset shape: {ntsb_final.shape}")
print(f"Columns remaining: {len(ntsb_final.columns)}")

Dropping 18 columns (>80% missing):
  ✗ child_restraint                (99.8% missing)
  ✗ vis_rvr                        (99.8% missing)
  ✗ med_type_flight                (99.3% missing)
  ✗ mr_faa_med_certf               (99.2% missing)
  ✗ mid_air                        (95.0% missing)
  ✗ on_ground_collision            (95.0% missing)
  ✗ oper_cert_num                  (94.5% missing)
  ✗ seat_occ_row                   (94.0% missing)
  ✗ oper_dba                       (92.2% missing)
  ✗ infl_rest_depl                 (91.6% missing)
  ✗ inj_s_grnd                     (89.4% missing)
  ✗ inj_f_grnd                     (89.2% missing)
  ✗ oper_code                      (88.5% missing)
  ✗ cc_seats                       (87.4% missing)
  ✗ inj_m_grnd                     (87.1% missing)
  ✗ oper_pax_cargo                 (84.0% missing)
  ✗ oper_dom_int                   (83.1% missing)
  ✗ oper_sched                     (82.2% missing)

Dataset shape: (48020, 182)
Columns remaining

Inspect target label "ev_highest_injury"

In [27]:
print("Target variable distribution (ev_highest_injury):")
print(ntsb_final['ev_highest_injury'].value_counts(dropna=False))
print(f"\nMissing: {ntsb_final['ev_highest_injury'].isna().sum()} rows")

# Drop rows where target is missing
ntsb_final = ntsb_final[ntsb_final['ev_highest_injury'].notna()].copy()
print(f"\nAfter dropping NaN targets: {ntsb_final.shape}")

Target variable distribution (ev_highest_injury):
ev_highest_injury
NONE    26275
FATL     7764
SERS     7595
MINR     6166
NaN       220
Name: count, dtype: int64

Missing: 220 rows

After dropping NaN targets: (47800, 182)


REMOVE DATA LEAKAGE COLUMNS

In [ ]:
leakage_cols = ['inj_tot_f', 'inj_tot_m', 'inj_tot_n', 'inj_tot_s']

print("Removing leakage columns :")
for col in leakage_cols:
    print(f"  ✗ {col}")

ntsb_final = ntsb_final.drop(columns=leakage_cols)
print(f"\nShape after leakage removal: {ntsb_final.shape}")

PROFILE THE 68 COMPLETE COLUMNS

In [30]:
complete_cols = [col for col in ntsb_final.columns 
                 if ntsb_final[col].isnull().sum() == 0]

print(f"Complete columns: {len(complete_cols)}\n")

# Separate by dtype
num_complete = [c for c in complete_cols if ntsb_final[c].dtype in ['int64','float64','int32']]
cat_complete = [c for c in complete_cols if ntsb_final[c].dtype == 'object']
bool_complete = [c for c in complete_cols if ntsb_final[c].dtype == 'bool']
date_complete = [c for c in complete_cols if 'datetime' in str(ntsb_final[c].dtype)]

print(f"  Numeric  : {len(num_complete)}")
print(f"  Categorical: {len(cat_complete)}")
print(f"  Boolean  : {len(bool_complete)}")
print(f"  DateTime : {len(date_complete)}")

# ── Numeric summary
print("\n── NUMERIC (complete) ──")
print(ntsb_final[num_complete].describe().round(2).T[
    ['mean','std','min','25%','50%','75%','max']
].to_string())

# ── Categorical: cardinality + top value
print("\n── CATEGORICAL (complete) ──")
for col in cat_complete:
    vc = ntsb_final[col].value_counts()
    print(f"  {col:35} unique={vc.nunique():>4}  top='{vc.index[0]}' ({vc.iloc[0]:,}, {vc.iloc[0]/len(ntsb_final)*100:.1f}%)")

# ── Boolean: % True
print("\n── BOOLEAN (complete) ──")
for col in bool_complete:
    pct = ntsb_final[col].sum() / len(ntsb_final) * 100
    print(f"  {col:35} True={ntsb_final[col].sum():>6,} ({pct:.1f}%)")

# ── DateTime
print("\n── DATETIME (complete) ──")
for col in date_complete:
    print(f"  {col:35} min={ntsb_final[col].min().date()}  max={ntsb_final[col].max().date()}")

Complete columns: 65

  Numeric  : 31
  Categorical: 22
  Boolean  : 11
  DateTime : 1

── NUMERIC (complete) ──
                       mean      std      min      25%      50%      75%        max
ev_year             2016.91     5.04  2008.00  2013.00  2017.00  2021.00    2025.00
ev_month               6.61     3.05     1.00     4.00     7.00     9.00      12.00
apt_dist              21.13   333.00     0.00     0.00     0.00     0.10    7470.00
wx_obs_dir           134.03   121.39    -2.00     0.00   120.00   236.00     360.00
wx_obs_dist            9.22    47.26   -19.00     0.00     2.00    12.00    6845.00
sky_nonceil_ht      1904.29  4375.07     0.00     0.00     0.00  2400.00  200000.00
sky_ceil_ht         1804.39  4672.70     0.00     0.00     0.00   400.00  260000.00
wx_temp               63.05    27.63   -71.00    52.00    68.00    81.00    3006.00
wx_dew_pt             44.67    65.75   -38.00    28.00    46.00    61.00    3027.00
wind_dir_deg         146.26   120.04     0.00  

CLEAN COMPLETE COLUMNS

In [31]:
# 1. DROP: IDs, audit cols, redundant, leakage, near-zero variance
cols_to_drop = [
    # identifiers / audit
    'ev_id', 'ntsb_no', 'ntsb_no_aircraft', 'latitude', 'longitude',
    'lchg_date', 'lchg_date_aircraft',
    # NOAA station metadata (not predictive)
    'noaa_station_id', 'noaa_station_name',
    # near-zero variance booleans
    'commercial_space_flight', 'noaa_hail', 'noaa_tornado',
    # near-constant categoricals
    'acft_missing',
    # redundant (already decoded into boolean cols)
    'noaa_frshtt',
    # leakage (total injuries encodes severity)
    'inj_tot_t',
]
ntsb_final.drop(columns=cols_to_drop, inplace=True)
print(f"After dropping useless cols: {ntsb_final.shape}")

# 2. FIX NOAA SENTINEL VALUES → NaN
# NOAA uses 999.9 / 9999.9 as "missing" placeholders
noaa_sentinels = {
    'noaa_dewp_f':      9999.9,
    'noaa_slp_hpa':     9999.9,
    'noaa_visib_miles': 999.9,
    'noaa_wind_knots':  999.9,
    'noaa_gust_knots':  999.9,
    'noaa_snow_in':     999.9,
    'noaa_prcp_in':     99.99,
    'noaa_maxwind_knots': 999.9,
    'noaa_temp_max_f':  9999.9,
    'noaa_temp_min_f':  9999.9,
    'noaa_temp_max_c':  9999.9,   # derived, same sentinel
    'noaa_temp_min_c':  9999.9,
}
for col, sentinel in noaa_sentinels.items():
    if col in ntsb_final.columns:
        n = (ntsb_final[col] == sentinel).sum()
        if n > 0:
            ntsb_final[col] = ntsb_final[col].replace(sentinel, np.nan)
            print(f"  {col}: {n:,} sentinels → NaN")

# 3. FIX PHYSICAL OUTLIERS → NaN (impossible real-world values)
physical_caps = {
    'wind_dir_deg': (0, 360),      # degrees can't exceed 360
    'wx_obs_dir':   (0, 360),
    'gust_kts':     (0, 150),      # no recorded wind > 150kts at ground level
    'altimeter':    (27, 32),      # normal range inHg
    'wx_temp':      (-80, 130),    # °F extremes on Earth
    'wx_dew_pt':    (-80, 100),
}
for col, (lo, hi) in physical_caps.items():
    if col in ntsb_final.columns:
        n = ((ntsb_final[col] < lo) | (ntsb_final[col] > hi)).sum()
        if n > 0:
            ntsb_final.loc[(ntsb_final[col] < lo) | (ntsb_final[col] > hi), col] = np.nan
            print(f"  {col}: {n:,} out-of-range values → NaN")

# 4. FEATURE ENGINEERING from ev_date
ntsb_final['ev_season'] = ntsb_final['ev_month'].map({
    12:'winter', 1:'winter', 2:'winter',
    3:'spring',  4:'spring', 5:'spring',
    6:'summer',  7:'summer', 8:'summer',
    9:'fall',   10:'fall',  11:'fall'
})
# ev_year and ev_month already exist, ev_dow already exists
ntsb_final.drop(columns=['ev_date'], inplace=True)
print(f"\nAfter all cleaning: {ntsb_final.shape}")

# 5. QUICK SANITY CHECK on NOAA cols post-fix
noaa_num_cols = ['noaa_temp_f', 'noaa_dewp_f', 'noaa_visib_miles', 
                 'noaa_wind_knots', 'noaa_gust_knots', 'noaa_slp_hpa']
print("\nNOAA numeric cols after sentinel fix:")
print(ntsb_final[[c for c in noaa_num_cols if c in ntsb_final.columns]].describe().round(2).T[['mean','min','max','count']])

After dropping useless cols: (47800, 163)
  noaa_dewp_f: 2,750 sentinels → NaN
  noaa_slp_hpa: 17,206 sentinels → NaN
  noaa_visib_miles: 3,435 sentinels → NaN
  noaa_wind_knots: 1,591 sentinels → NaN
  noaa_gust_knots: 21,071 sentinels → NaN
  noaa_snow_in: 47,117 sentinels → NaN
  noaa_prcp_in: 2,603 sentinels → NaN
  noaa_maxwind_knots: 1,741 sentinels → NaN
  noaa_temp_max_f: 14 sentinels → NaN
  noaa_temp_min_f: 3 sentinels → NaN
  wind_dir_deg: 2 out-of-range values → NaN
  wx_obs_dir: 14 out-of-range values → NaN
  gust_kts: 2 out-of-range values → NaN
  altimeter: 4,017 out-of-range values → NaN
  wx_temp: 42 out-of-range values → NaN
  wx_dew_pt: 44 out-of-range values → NaN

After all cleaning: (47800, 163)

NOAA numeric cols after sentinel fix:
                     mean    min     max    count
noaa_temp_f         62.05  -43.8   104.9  47800.0
noaa_dewp_f         46.81  -36.3    84.1  45050.0
noaa_visib_miles     9.60    0.0    77.2  44365.0
noaa_wind_knots      5.55    0.0  

In [33]:
print(ntsb_final['noaa_gust_knots'].value_counts().head(20))

noaa_gust_knots
18.1    2046
15.9    1994
17.1    1926
20.0    1896
21.0    1889
19.0    1782
15.0    1670
22.0    1666
22.9    1416
14.0    1330
24.1    1318
26.0     907
27.0     900
25.1     890
28.0     730
28.9     548
29.9     461
32.1     406
31.1     354
9.9      331
Name: count, dtype: int64


PROFILE EASY-IMPUTE COLUMNS (<20% missing)

In [34]:
easy_impute_cols = [col for col in ntsb_final.columns
                    if 0 < ntsb_final[col].isnull().mean() < 0.20]

print(f"Easy-impute columns: {len(easy_impute_cols)}\n")

num_easy = [c for c in easy_impute_cols if ntsb_final[c].dtype in ['float64','int64','int32']]
cat_easy  = [c for c in easy_impute_cols if ntsb_final[c].dtype == 'object']

print(f"  Numeric : {len(num_easy)}")
print(f"  Categorical: {len(cat_easy)}")

# Numeric: missing count + basic stats
print("\n── NUMERIC (<20% missing) ──")
for col in num_easy:
    pct  = ntsb_final[col].isnull().mean() * 100
    med  = ntsb_final[col].median()
    mn   = ntsb_final[col].min()
    mx   = ntsb_final[col].max()
    print(f"  {col:30} missing={pct:5.1f}%  median={med:8.2f}  min={mn:8.2f}  max={mx:8.2f}")

# Categorical: missing count + cardinality + top value
print("\n── CATEGORICAL (<20% missing) ──")
for col in cat_easy:
    pct     = ntsb_final[col].isnull().mean() * 100
    nuniq   = ntsb_final[col].nunique()
    top_val = ntsb_final[col].value_counts().index[0]
    top_pct = ntsb_final[col].value_counts().iloc[0] / ntsb_final[col].notna().sum() * 100
    print(f"  {col:30} missing={pct:5.1f}%  unique={nuniq:5}  top='{top_val}' ({top_pct:.1f}%)")

Easy-impute columns: 78

  Numeric : 22
  Categorical: 56

── NUMERIC (<20% missing) ──
  ev_time                        missing=  0.1%  median= 1730.00  min=    0.00  max= 2359.00
  wx_obs_time                    missing=  8.5%  median= 1600.00  min=    0.00  max= 2359.00
  wx_obs_dir                     missing=  0.0%  median=  120.00  min=    0.00  max=  360.00
  wx_obs_elev                    missing= 15.3%  median=  620.00  min= -115.00  max=751635.00
  vis_sm                         missing=  5.3%  median=   10.00  min=    0.00  max= 5000.00
  wx_temp                        missing=  0.1%  median=   68.00  min=  -71.00  max=  122.00
  wx_dew_pt                      missing=  0.1%  median=   46.00  min=  -38.00  max=  100.00
  wind_dir_deg                   missing=  0.0%  median=  150.00  min=    0.00  max=  360.00
  gust_kts                       missing=  0.0%  median=    0.00  min=    0.00  max=   57.00
  altimeter                      missing=  8.4%  median=   30.02  min=   2

EASY-IMPUTE COLUMNS: DROP, CAP, THEN IMPUTE

In [36]:
import warnings
warnings.filterwarnings("ignore")

# 1. DROP: admin, IDs, constants, audit timestamps, free text
drop_easy = [
    # single value / near-constant
    'ev_tmzn', 'certs_held', 'invest_agy',
    # audit timestamps
    'lchg_date_crew', 'lchg_date_narr', 'date_last_insp',
    # free text narratives (unusable by tree models)
    'narr_accf', 'narr_cause',
    # location admin / high-cardinality IDs with no ML signal
    'ev_site_zipcode', 'wx_obs_fac_id', 'owner_zip', 'oper_zip',
    'dprt_apt_id', 'dprt_city', 'dprt_state', 'dprt_country',
    'dest_country', 'crew_city', 'owner_city', 'oper_city',
    'acft_serial_no', 'regis_no',
    # redundant geography
    'owner_state', 'oper_state', 'crew_res_state',
    'crew_res_country', 'oper_country', 'owner_country',
]
drop_easy = [c for c in drop_easy if c in ntsb_final.columns]
ntsb_final.drop(columns=drop_easy, inplace=True)
print(f"After dropping admin/ID easy cols: {ntsb_final.shape}")

# 2. CAP impossible outliers BEFORE imputing
# afm_hrs: cap at 50,000 hrs (realistic max for a well-used aircraft)
ntsb_final.loc[ntsb_final['afm_hrs'] > 50000, 'afm_hrs'] = np.nan
# wx_obs_elev: cap at 15,000 ft (highest US airports ~7,000 ft)
ntsb_final.loc[ntsb_final['wx_obs_elev'] > 15000, 'wx_obs_elev'] = np.nan
# vis_sm: cap at 50 miles (standard "unlimited" visibility)
ntsb_final.loc[ntsb_final['vis_sm'] > 50, 'vis_sm'] = np.nan
# cert_max_gr_wt: cap at 900,000 lbs (Antonov AN-225 max)
ntsb_final.loc[ntsb_final['cert_max_gr_wt'] > 900000, 'cert_max_gr_wt'] = np.nan
print("Outlier capping done")

# 3. NUMERIC IMPUTATION → median
num_impute_median = [
    'ev_time', 'wx_obs_time', 'wx_obs_dir', 'wx_obs_elev',
    'vis_sm', 'altimeter', 'cert_max_gr_wt', 'total_seats',
    'num_eng', 'afm_hrs', 'crew_no',
    'noaa_dewp_f', 'noaa_visib_miles', 'noaa_wind_knots',
    'noaa_maxwind_knots', 'noaa_prcp_in',
]
for col in num_impute_median:
    if col in ntsb_final.columns:
        median_val = ntsb_final[col].median()
        ntsb_final[col].fillna(median_val, inplace=True)

# 4. CATEGORICAL IMPUTATION → mode or 'UNKNOWN'
mode_impute = [
    'light_cond', 'wx_cond_basic', 'sky_cond_nonceil', 'sky_cond_ceil',
    'damage', 'acft_fire', 'acft_expl',
    'far_part', 'flt_plan_filed', 'type_fly', 'type_last_insp', 'elt_install',
    'crew_category', 'crew_inj_level', 'seat_occ_pic', 'pilot_flying',
    'second_pilot', 'pc_profession',
    'ev_state', 'ev_city', 'ev_nr_apt_loc', 'latlong_acq', 'wx_src_iic',
    'air_medical', 'site_seeing',
]
for col in mode_impute:
    if col in ntsb_final.columns:
        mode_val = ntsb_final[col].mode()[0]
        ntsb_final[col].fillna(mode_val, inplace=True)

# high-cardinality → 'UNKNOWN' instead of mode
for col in ['acft_make', 'acft_model', 'acft_category']:
    if col in ntsb_final.columns:
        ntsb_final[col].fillna('UNKNOWN', inplace=True)

# 5. FINAL CHECK
remaining_nan = ntsb_final.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0].sort_values(ascending=False)
print(f"\nFinal shape: {ntsb_final.shape}")
print(f"Columns still with NaN: {len(remaining_nan)}")
print(remaining_nan.head(20))

After dropping admin/ID easy cols: (47800, 135)
Outlier capping done

Final shape: (47800, 135)
Columns still with NaN: 57
noaa_snow_in            47117
dprt_pt_same_ev         37240
metar                   35306
med_crtf_limit          34797
elt_aided_loc_ev        32913
oper_street             32784
afm_hrs_last_insp       32617
ft_as_of                31787
elt_model               30623
owner_street            28910
elt_manufacturer        28897
acft_series             28505
lchg_userid_crew        27269
lchg_userid_narr        26611
lchg_userid_aircraft    26550
lchg_userid             26550
apt_dir                 25802
fuel_on_board           25548
bfr_date                22601
pax_seats               22541
dtype: int64


IMPUTE (20-50%) AND REVIEW (50-80%) TIERS

In [38]:
# GROUP 1 — DROP: audit userids, street addresses, admin text
drop_impute = [
    'lchg_userid_crew', 'lchg_userid_narr',
    'lchg_userid_aircraft', 'lchg_userid',  # audit user IDs
    'oper_street', 'owner_street',           # street addresses, no ML value
    'metar',                                 # raw weather text string
    'ft_as_of',                              # "flight time as of" date string
    'bfr_date',                              # biennial flight review date
    'date_lst_med',                          # last medical date
    'dest_apt_id', 'apt_name',               # destination admin
    'oper_name',                             # operator name text
    'narr_accp',                             # accident report narrative (text)
    'dest_city', 'dest_state',               # redundant geography
]
drop_impute = [c for c in drop_impute if c in ntsb_final.columns]
ntsb_final.drop(columns=drop_impute, inplace=True)
print(f"After dropping impute-tier admin cols: {ntsb_final.shape}")

# GROUP 2 — DROP: noaa_snow_in (47,117 NaN = 98.6% missing after sentinel fix)
# This column was already gutted by sentinel replacement — almost nothing left
ntsb_final.drop(columns=['noaa_snow_in'], inplace=True, errors='ignore')

# GROUP 3 — MEDIAN impute: numeric cols with real signal
num_median_impute = [
    'pax_seats',        # passenger seats — aircraft size proxy
    'rwy_width',        # runway dimensions — landing context
    'rwy_len',
    'fc_seats',         # flight crew seats
    'apt_elev',         # airport elevation
    'acft_year',        # aircraft manufacture year
    'dprt_time',        # departure time
    'wind_vel_kts',     # wind speed at event
    'crew_age',         # pilot age
    'afm_hrs_last_insp', # hours since last inspection
    'fuel_on_board',    # fuel remaining
    'apt_dir',          # airport direction
]
for col in num_median_impute:
    if col in ntsb_final.columns:
        ntsb_final[col] = ntsb_final[col].fillna(ntsb_final[col].median())
print("Numeric median imputation done")

# GROUP 4 — MODE impute: low-cardinality categoricals with real signal
mode_impute2 = [
    'rwy_num',              # runway number
    'elt_type',             # emergency locator transmitter type
    'wx_obs_tmzn',          # weather obs timezone
    'owner_acft',           # owner type
    'infl_rest_inst',       # inflatable restraint installed
    'restraint_used',       # restraint used
    'available_restraint',  # restraint available
    'crew_tox_perf',        # toxicology performed
    'elt_oper',             # ELT operational
    'med_crtf_vldty',       # medical certificate validity
    'ev_nr_apt_id',         # nearest airport
    'flight_plan_activated','crew_sex',
    'med_certf',            # medical certificate type
]
for col in mode_impute2:
    if col in ntsb_final.columns:
        ntsb_final[col] = ntsb_final[col].fillna(ntsb_final[col].mode()[0])

# GROUP 5 — REVIEW tier decisions
# dprt_pt_same_ev (77.9%): "did departure point equal event location?" — keep, fill 'UNKN'
# med_crtf_limit (72.9%): medical certificate limitations — fill 'NONE'
# elt_aided_loc_ev (68.9%): ELT aided location of event — fill mode
# acft_series (59.6%): aircraft series — fill 'UNKNOWN'
# elt_model / elt_manufacturer (60-64%): fill 'UNKNOWN'
review_fills = {
    'dprt_pt_same_ev':   'UNKN',
    'med_crtf_limit':    'NONE',
    'elt_aided_loc_ev':  'N',
    'acft_series':       'UNKNOWN',
    'elt_model':         'UNKNOWN',
    'elt_manufacturer':  'UNKNOWN',
}
for col, fill_val in review_fills.items():
    if col in ntsb_final.columns:
        ntsb_final[col] = ntsb_final[col].fillna(fill_val)

remaining = ntsb_final.isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print(f"\nFinal shape: {ntsb_final.shape}")
print(f"Columns still with NaN: {len(remaining)}")
if len(remaining) > 0:
    print(remaining)
else:
    print("Zero NaN columns — dataset is clean!")

After dropping impute-tier admin cols: (47800, 118)
Numeric median imputation done

Final shape: (47800, 118)
Columns still with NaN: 8
noaa_gust_knots    21071
noaa_slp_hpa       17206
wx_dew_pt             44
wx_temp               42
noaa_temp_max_f       14
noaa_temp_min_f        3
gust_kts               2
wind_dir_deg           2
dtype: int64


FINAL NaN CLEANUP — 8 remaining columns

In [40]:
final_nan_cols = [
    'noaa_gust_knots', 'noaa_slp_hpa', 'wx_dew_pt',
    'wx_temp', 'noaa_temp_max_f', 'noaa_temp_min_f',
    'gust_kts', 'wind_dir_deg'
]

for col in final_nan_cols:
    median_val = ntsb_final[col].median()
    ntsb_final[col] = ntsb_final[col].fillna(median_val)
    print(f"  {col:25} → filled with median {median_val:.2f}")

total_nan = ntsb_final.isnull().sum().sum()
print(f"\nTotal NaN remaining: {total_nan}")
print(f"Final clean shape: {ntsb_final.shape}")

output_path = '../data/processed/ntsb_clean_final.csv'
ntsb_final.to_csv(output_path, index=False)
print(f"\nDataset saved to {output_path}")

  noaa_gust_knots           → filled with median 21.00
  noaa_slp_hpa              → filled with median 1015.70
  wx_dew_pt                 → filled with median 46.00
  wx_temp                   → filled with median 68.00
  noaa_temp_max_f           → filled with median 78.10
  noaa_temp_min_f           → filled with median 52.20
  gust_kts                  → filled with median 0.00
  wind_dir_deg              → filled with median 150.00

Total NaN remaining: 0
Final clean shape: (47800, 118)

Dataset saved to ../data/processed/ntsb_clean_final.csv


Final Output of EDA :
Started with 200 columns → dropped 82 (IDs, leakage, constants, admin, high-missing). 
Fixed NOAA sentinel values (999.9 / 9999.9) masquerading as real data across 10 columns. 
Capped physically impossible outliers in 6 columns. 
Imputed all remaining NaN with median (numeric) or mode / 'UNKNOWN' (categorical). 
Extracted temporal features from ev_date (month, season, year). Removed data leakage columns (inj_tot_*). 
Saved ntsb_clean_final.csv